# 03 Frozen embedding Logistic Regression baseline

Extract fixed ImageNet ResNet-18 features for each fixed split, mean-pool within study, and select Logistic Regression C using validation only. Test predictions are written only after selection has frozen; final test metrics are calculated later in notebook 05.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = os.environ.get("MURA_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("Set MURA_DATA_ROOT first.")

manifest = ROOT / "data/manifests/wrist_manifest.csv"
split_manifest = ROOT / "data/manifests/split_manifest.csv"
processed = ROOT / "data/processed"
processed.mkdir(parents=True, exist_ok=True)


In [ ]:
for split in ("train", "val", "test"):
    subprocess.run(
        [
            sys.executable, "scripts/extract_embeddings.py",
            "--manifest", str(manifest),
            "--split-manifest", str(split_manifest),
            "--split", split,
            "--output", str(processed / f"embeddings_{split}.npz"),
            "--dataset-root", DATA_ROOT,
            "--allow-weight-download",
        ],
        cwd=ROOT,
        check=True,
    )


In [ ]:
subprocess.run(
    [
        sys.executable, "scripts/train_logistic_baseline.py",
        "--train-embeddings", str(processed / "embeddings_train.npz"),
        "--val-embeddings", str(processed / "embeddings_val.npz"),
        "--test-embeddings", str(processed / "embeddings_test.npz"),
        "--write-test-predictions",
        "--output-dir", "models/frozen_logistic",
    ],
    cwd=ROOT,
    check=True,
)


In [ ]:
import json
selection_path = ROOT / "models/frozen_logistic/selection.json"
print(json.loads(selection_path.read_text(encoding="utf-8")))
